## Week 2 Day 1

And now! Our first look at OpenAI Agents SDK

You won't believe how lightweight this is..

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/tools.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#00bfff;">The OpenAI Agents SDK Docs</h2>
            <span style="color:#00bfff;">The documentation on OpenAI Agents SDK is really clear and simple: <a href="https://openai.github.io/openai-agents-python/">https://openai.github.io/openai-agents-python/</a> and it's well worth a look.
            </span>
        </td>
    </tr>
</table>

# Three Parts to this lab

## Part 1: A simple "Agent" and "Agent Loop"

Basically an LLM call. We'll add tracing and streaming to the mix.

## Part 2: Adding a Tool

A familiar one, but oh-so-easy

## Part 3: Adding Memory

So that different Agent calls know about each other

In [ ]:
# The imports
print("Venu")
import os
import requests
from dotenv import load_dotenv
from openai.types.responses import ResponseTextDeltaEvent
from agents import Agent, Runner, trace, function_tool, SQLiteSession
load_dotenv(override=True)


## Sidenote

The actual name of this framework on the official Python index pypi.org is `openai-agents`

So for your own projects in the future, you would do:

`pip install openai-agents`  
or  
`uv add openai-agents`

followed by

`from agents import Agent, Runner, trace`

Beware that doing a `pip install agents` would install something completely different - an older reinforcement learning library.


In [ ]:

# Make an agent with name, instructions, model

agent = Agent(name="Jokester", instructions="You are a joke teller", model="gpt-5.4-mini")

In [ ]:
# Run the joke with Runner.run(agent, prompt)

result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")


In [ ]:
# Here is the final output

print(result.final_output)

In [ ]:
# Here is the detail of the LLM calls

result.to_input_list()

## Adding Observability with a trace

In [ ]:
with trace("Telling a joke"):
    result = await Runner.run(agent, "Tell a joke about Autonomous AI Agents")
print(result.final_output)

## Now go and look at the trace

https://platform.openai.com/traces

In [ ]:
# Streaming

result = Runner.run_streamed(agent, input="Please tell me 5 jokes about AI Agents.")
async for event in result.stream_events():
    if event.type == "raw_response_event" and isinstance(event.data, ResponseTextDeltaEvent):
        print(event.data.delta, end="", flush=True)

## Part 2: Adding a tool

In [ ]:
pushover_user = os.getenv("PUSHOVER_USER")
pushover_token = os.getenv("PUSHOVER_TOKEN")
pushover_url = "https://api.pushover.net/1/messages.json"

if pushover_user:
    if pushover_user.startswith("u"):
        print("Pushover user found and looks good")
    else:
        print("Pushover user found but doesn't start with u")
else:
    print("Pushover user not found")

if pushover_token:
    if pushover_token.startswith("a"):
        print("Pushover token found and looks good")
    else:
        print("Pushover token found but doesn't start with a")
else:
    print("Pushover token not found")

In [ ]:
# Remember this?

def push(message):
    print(f"Push: {message}")
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    requests.post(pushover_url, data=payload)

In [ ]:
push("HEY!!")

In [ ]:
push

In [ ]:
# Now this:

@function_tool
def push_tool(message: str) -> str:
    """ Send the given message to the user as a push notification """
    payload = {"user": pushover_user, "token": pushover_token, "message": message}
    result = requests.post(pushover_url, data=payload).status_code
    return f"Push sent with API status code {result}"

In [ ]:
push_tool

In [ ]:
push_tool.description

In [ ]:

notifier = Agent(name="Notifier", model="gpt-5.4-mini", instructions="You notify the user upon request", tools=[push_tool])

In [ ]:
with trace("Pizza has arrived"):
    result = await Runner.run(notifier, "Notify the user that the pizza is here")

print(result.final_output)


## Now go and look at the trace

https://platform.openai.com/traces

## Part 3: Sessions (memory)

Within a Runner.run() application level turn, the conversation history is maintained.

But each call to Runner.run() is a fresh start.

Let's see that:

In [ ]:
agent = Agent(name="Assistant", model="gpt-5.4-mini")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?")
print(response.final_output)

## Memory approach 1 - just manually pass in the list of dicts

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.")
print(response.final_output)

In [ ]:
response.to_input_list()

In [ ]:
next_input = response.to_input_list() + [{"role": "user", "content": "What's my name?"}]
next_input

In [ ]:
response = await Runner.run(agent, next_input)
print(response.final_output)

## Another approach - use OpenAI Agents SDK built in SQLLite session

In [ ]:
# This is created in-memory
# For an on-disk memory, use SQLiteSession("12345", "memory.db")

session = SQLiteSession("12346")

In [ ]:
response = await Runner.run(agent, "Hi there. My name is Ed.", session=session)
print(response.final_output)

In [ ]:
response = await Runner.run(agent, "What's my name?", session=session)
print(response.final_output)

# WOW

Can you believe how much we got done in Lab 1?!

Agents, Runner (Agent Loop), traces (Observability), Streaming, Function Tools, Memory!

Remember to check out the docs:  
https://openai.github.io/openai-agents-python/

Even better news: many of the lightweight Agent Frameworks are very similar, so you practically know them all..


<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Make one of the Week 1 projects using OpenAI Agents SDK - like the digital twin or the Checklist loop. You will be astonished how easy it is.
            </span>
        </td>
    </tr>
</table>